<a href="https://colab.research.google.com/github/Danyribeiro/API-controle-de-ponto-e-acesso/blob/main/TCC_qualidade_dos_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
!pip install great_expectations statsmodels -q
import statsmodels.api as sm
import great_expectations as gx
import warnings

warnings.filterwarnings("ignore")

# ==========================================================
# 1. LEITURA E PRÉ-PROCESSAMENTO DOS DADOS
# ==========================================================

def ler_csv_seguro(caminho, separador=","):
    """Tenta ler CSV com detecção automática de cabeçalho."""
    try:
        df = pd.read_csv(caminho, sep=separador, encoding="utf-8-sig", engine="python")
        if df.shape[1] == 1:
            df = pd.read_csv(caminho, sep=separador, encoding="utf-8-sig",
                             header=None, skiprows=1, engine="python")
        return df
    except Exception as e:
        raise Exception(f"Erro ao ler {caminho}: {e}")

pib   = ler_csv_seguro("pib.csv", ",")           # ✅ sem /
pop   = ler_csv_seguro("populacao.csv", ",")      # ✅ sem /
analf = ler_csv_seguro("analfabetismo.csv", ";")  # ✅ sem /

pib   = pib.iloc[:, :4].copy()
pop   = pop.iloc[:, :4].copy()
analf = analf.iloc[:, :4].copy()

pib.columns   = ["Sigla", "Codigo", "Municipio", "PIB_2021"]
pop.columns   = ["Sigla", "Codigo", "Municipio", "Populacao_2021"]
analf.columns = ["Sigla", "Codigo", "Municipio", "Analfabetismo_2022"]

# ==========================================================
# 2. LIMPEZA DOS CÓDIGOS MUNICIPAIS
# ==========================================================

def limpar_codigo(df):
    df["Codigo"] = (
        df["Codigo"]
        .astype(str)
        .str.extract(r"(\d+)")[0]
        .str.zfill(7)
    )
    df = df[df["Codigo"].str.match(r"^\d{7}$", na=False)]
    return df

pib   = limpar_codigo(pib)
pop   = limpar_codigo(pop)
analf = limpar_codigo(analf)

# ==========================================================
# 3. CONVERSÃO DOS TIPOS
# ==========================================================

pib["PIB_2021"] = pd.to_numeric(pib["PIB_2021"], errors="coerce")
pop["Populacao_2021"] = pd.to_numeric(pop["Populacao_2021"], errors="coerce")
analf["Analfabetismo_2022"] = (
    analf["Analfabetismo_2022"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

# ==========================================================
# 4. INTEGRAÇÃO DAS BASES
# ==========================================================

base = pib.merge(pop[["Codigo", "Populacao_2021"]], on="Codigo", how="inner")
base = base.merge(analf[["Codigo", "Analfabetismo_2022"]], on="Codigo", how="inner")
base = base.replace([np.inf, -np.inf], np.nan)

print(f"Total de municípios na base integrada: {len(base)}")

# ==========================================================
# 5. CONFIGURAÇÃO DO GREAT EXPECTATIONS (GX)
# ==========================================================

context          = gx.get_context(mode="ephemeral")
data_source      = context.data_sources.add_pandas(name="dados_municipais")
data_asset       = data_source.add_dataframe_asset(name="base_municipios")
batch_definition = data_asset.add_batch_definition_whole_dataframe("municipios_batch")
batch            = batch_definition.get_batch(batch_parameters={"dataframe": base})

# ==========================================================
# 6. COMPLETUDE (AVALIAÇÃO)
# ==========================================================

expect_completude = [
    gx.expectations.ExpectColumnValuesToNotBeNull(column=col)
    for col in ["PIB_2021", "Populacao_2021", "Analfabetismo_2022"]
]

# ==========================================================
# 7. VALIDADE (AVALIAÇÃO)
# ==========================================================

expect_validade = [
    gx.expectations.ExpectColumnValuesToBeBetween(column="PIB_2021", min_value=0),
    gx.expectations.ExpectColumnValuesToBeBetween(column="Populacao_2021", min_value=1),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Analfabetismo_2022", min_value=0, max_value=100
    )
]

todas_expectativas = expect_completude + expect_validade

print("\n===== RESULTADOS DAS EXPECTATIONS =====")

missing_count   = {}
invalidos_count = {}

for exp in todas_expectativas:
    resultado = batch.validate(exp)
    nome      = exp.__class__.__name__
    col       = exp.column
    print(f"\n{nome} - {col}")
    print(f"Sucesso: {resultado.success}")
    print(resultado.result)
    n_inesperados = resultado.result.get("unexpected_count", 0)
    if nome == "ExpectColumnValuesToNotBeNull":
        missing_count[col] = n_inesperados
    elif nome == "ExpectColumnValuesToBeBetween":
        invalidos_count[col] = n_inesperados

# ==========================================================
# 8. CONSISTÊNCIA (DETECÇÃO DE OUTLIERS POR IQR)
# ==========================================================

print("\n===== DETECÇÃO DE OUTLIERS (IQR) =====")

limites_iqr       = {}
outliers_detalhes = {}

for col in ["PIB_2021", "Populacao_2021", "Analfabetismo_2022"]:
    q1  = base[col].quantile(0.25)
    q3  = base[col].quantile(0.75)
    iqr = q3 - q1
    li  = q1 - 1.5 * iqr
    ls  = q3 + 1.5 * iqr
    outliers = base[(base[col] < li) | (base[col] > ls)]
    limites_iqr[col]       = {"inferior": li, "superior": ls, "n_outliers": len(outliers)}
    outliers_detalhes[col] = outliers[["Municipio", col]].copy()
    print(f"\n{col}: {len(outliers)} outliers | Limites: [{li:.2f}, {ls:.2f}]")

# ==========================================================
# 9. ANÁLISE EXPLORATÓRIA DOS OUTLIERS (TOP 5 E BOTTOM 5)
# ==========================================================

print("\n===== ANÁLISE EXPLORATÓRIA DOS OUTLIERS =====")

for col in ["PIB_2021", "Populacao_2021", "Analfabetismo_2022"]:
    if len(outliers_detalhes[col]) > 0:
        top5 = outliers_detalhes[col].sort_values(col, ascending=False).head(5)
        print(f"\nTop 5 outliers de {col}:")
        print(top5.to_string(index=False))

        bottom5 = outliers_detalhes[col].sort_values(col, ascending=True).head(5)
        print(f"\nBottom 5 outliers de {col}:")
        print(bottom5.to_string(index=False))

print("\n→ Decisão sobre outliers:")
print("  Após inspeção dos extremos superiores e inferiores, os valores foram")
print("  considerados observações legítimas da população estudada (municípios")
print("  de grande e pequeno porte), não caracterizando erros de registro.")
print("  Portanto, todos os outliers foram MANTIDOS na análise.")

# ==========================================================
# 10. TRATAMENTO DAS INCONSISTÊNCIAS
# ==========================================================

base_tratada = base.copy()

# 10.1 COMPLETUDE
print("\n===== TRATAMENTO — COMPLETUDE =====")

missing_original = {
    col: (base[col].isna().sum(), base[col].isna().sum() / len(base) * 100)
    for col in ["PIB_2021", "Populacao_2021", "Analfabetismo_2022"]
}

cols_imputar = [col for col, (n, pct) in missing_original.items() if pct >= 5]
cols_excluir = [col for col, (n, pct) in missing_original.items() if 0 < pct < 5]

for col in cols_imputar:
    n, pct  = missing_original[col]
    mediana = base_tratada[col].median()
    base_tratada[col].fillna(mediana, inplace=True)
    print(f"{col}: {n} ausentes ({pct:.2f}%) >= 5% → imputado pela mediana = {mediana:.4f}")

if cols_excluir:
    n_antes    = len(base_tratada)
    base_tratada = base_tratada.dropna(subset=cols_excluir)
    n_removidos  = n_antes - len(base_tratada)
    print(f"Exclusão em bloco de {n_removidos} registros com missing nas colunas: {', '.join(cols_excluir)}")
else:
    print("Nenhuma coluna com missing < 5% para exclusão.")

if not cols_imputar and not cols_excluir:
    print("Nenhum valor ausente encontrado — nenhuma ação necessária.")

# 10.2 VALIDADE
print("\n===== TRATAMENTO — VALIDADE =====")

regras = (
    (base_tratada["Analfabetismo_2022"] >= 0) &
    (base_tratada["Analfabetismo_2022"] <= 100) &
    (base_tratada["Populacao_2021"] > 0) &
    (base_tratada["PIB_2021"] >= 0)
)
n_invalidos = (~regras).sum()
if n_invalidos > 0:
    base_tratada = base_tratada[regras].copy()
    print(f"{n_invalidos} registros removidos por valores inválidos.")
else:
    print("Nenhum registro inválido encontrado.")

# 10.3 CONSISTÊNCIA
print("\n===== TRATAMENTO — CONSISTÊNCIA =====")
for col in ["PIB_2021", "Populacao_2021", "Analfabetismo_2022"]:
    info = limites_iqr[col]
    print(
        f"{col}: {info['n_outliers']} outliers detectados "
        f"→ MANTIDOS (após análise exploratória de topo e base)"
    )

print(f"\nBase tratada final: {len(base_tratada)} municípios.")

# ==========================================================
# 11. REGRESSÃO MQO
# ==========================================================

def rodar_regressao(df, nome_cenario):
    X      = sm.add_constant(df[["Populacao_2021", "Analfabetismo_2022"]])
    y      = df["PIB_2021"]
    modelo = sm.OLS(y, X, missing="drop").fit()

    print(f"\n{'='*60}\nREGRESSÃO MQO — {nome_cenario}\n{'='*60}")
    print(f"Observações : {int(modelo.nobs)}")
    print(f"R²          : {modelo.rsquared:.6f}")
    print(f"R² ajustado : {modelo.rsquared_adj:.6f}")
    print(f"F-statistic : {modelo.fvalue:.4f}  (p={modelo.f_pvalue:.4e})")
    print("\nCoeficientes:")
    for var in modelo.params.index:
        coef = modelo.params[var]
        pval = modelo.pvalues[var]
        sig  = (
            "***" if pval < 0.001 else
            "**"  if pval < 0.01  else
            "*"   if pval < 0.05  else
            "n.s."
        )
        print(f"  {var:<22}: {coef:>16.4f}   p={pval:.4e}  {sig}")
    return modelo

# ==========================================================
# 12. ESTIMAÇÃO NOS DOIS CENÁRIOS
# ==========================================================

modelo_bruto   = rodar_regressao(base.copy(), "DADOS BRUTOS (sem tratamento)")
modelo_tratado = rodar_regressao(base_tratada, "DADOS TRATADOS")

# ==========================================================
# 13. ANÁLISE COMPARATIVA
# ==========================================================

if modelo_bruto is not None and modelo_tratado is not None:

    print(f"\n{'='*60}")
    print("ANÁLISE COMPARATIVA — BRUTO vs TRATADO")
    print(f"{'='*60}")

    for label, attr in [
        ("Observações",  "nobs"),
        ("R²",           "rsquared"),
        ("R² ajustado",  "rsquared_adj"),
        ("F-statistic",  "fvalue"),
    ]:
        val_b = getattr(modelo_bruto,   attr)
        val_t = getattr(modelo_tratado, attr)
        print(
            f"  {label:<20}: "
            f"Bruto={val_b:>12.6g}  "
            f"Tratado={val_t:>12.6g}  "
            f"Δ={val_t - val_b:>+.6g}"
        )

    variaveis  = ["const", "Populacao_2021", "Analfabetismo_2022"]
    comparacao = pd.DataFrame({
        "Variável":        variaveis,
        "Coef_bruto":      [modelo_bruto.params.get(v, np.nan)   for v in variaveis],
        "p-valor_bruto":   [modelo_bruto.pvalues.get(v, np.nan)  for v in variaveis],
        "Coef_tratado":    [modelo_tratado.params.get(v, np.nan) for v in variaveis],
        "p-valor_tratado": [modelo_tratado.pvalues.get(v, np.nan) for v in variaveis],
    })
    print("\nCoeficientes e p-valores:")
    print(comparacao.to_string(index=False))

    print("\n===== INTERPRETAÇÃO DOS RESULTADOS =====")
    if abs(modelo_bruto.rsquared - modelo_tratado.rsquared) < 0.0001:
        print(
            "O tratamento das inconsistências identificadas não alterou "
            "substancialmente o poder explicativo do modelo. Isso sugere "
            "que os problemas de qualidade encontrados apresentavam baixa "
            "influência sobre as estimativas da regressão."
        )
    else:
        print(
            "O tratamento das inconsistências produziu alterações nos "
            "resultados do modelo, indicando impacto da qualidade dos dados "
            "sobre as estimativas obtidas."
        )

print("\n✔ Pipeline concluído com total conformidade à metodologia.")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp7615tn0p' for ephemeral docs site


Total de municípios na base integrada: 5597

===== RESULTADOS DAS EXPECTATIONS =====


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]


ExpectColumnValuesToNotBeNull - PIB_2021
Sucesso: False
{'element_count': 5597, 'unexpected_count': 27, 'unexpected_percent': 0.4824012864034304, 'partial_unexpected_list': [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan], 'partial_unexpected_counts': [{'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}, {'value': nan, 'count': 1}], 'partial_unexpected_index_list': [619, 620, 621, 622, 757, 808, 809, 810, 890, 891, 2304, 2429, 2574, 2575, 2984, 2985, 2986, 3610, 3

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

...


ExpectColumnValuesToBeBetween - PIB_2021
Sucesso: True
{'element_count': 5597, 'unexpected_count': 0, 'unexpected_percent': 0.0, 'partial_unexpected_list': [], 'missing_count': 27, 'missing_percent': 0.4824012864034304, 'unexpected_percent_total': 0.0, 'unexpected_percent_nonmissing': 0.0, 'partial_unexpected_counts': [], 'partial_unexpected_index_list': []}


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]


ExpectColumnValuesToBeBetween - Populacao_2021
Sucesso: True
{'element_count': 5597, 'unexpected_count': 0, 'unexpected_percent': 0.0, 'partial_unexpected_list': [], 'missing_count': 27, 'missing_percent': 0.4824012864034304, 'unexpected_percent_total': 0.0, 'unexpected_percent_nonmissing': 0.0, 'partial_unexpected_counts': [], 'partial_unexpected_index_list': []}


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]


ExpectColumnValuesToBeBetween - Analfabetismo_2022
Sucesso: True
{'element_count': 5597, 'unexpected_count': 0, 'unexpected_percent': 0.0, 'partial_unexpected_list': [], 'missing_count': 27, 'missing_percent': 0.4824012864034304, 'unexpected_percent_total': 0.0, 'unexpected_percent_nonmissing': 0.0, 'partial_unexpected_counts': [], 'partial_unexpected_index_list': []}

===== DETECÇÃO DE OUTLIERS (IQR) =====

PIB_2021: 734 outliers | Limites: [-345361.42, 715813.85]

Populacao_2021: 602 outliers | Limites: [-25012.12, 56230.88]

Analfabetismo_2022: 1 outliers | Limites: [-13.05, 36.57]

===== ANÁLISE EXPLORATÓRIA DOS OUTLIERS =====

Top 5 outliers de PIB_2021:
     Municipio     PIB_2021
     São Paulo 3.869743e+08
Rio de Janeiro 1.678791e+08
      Brasília 1.339474e+08
Belo Horizonte 4.944463e+07
        Manaus 4.821251e+07

Bottom 5 outliers de PIB_2021:
        Municipio      PIB_2021
             Ibiá 716373.669453
       Coromandel 716643.950873
   Arroio do Meio 717277.408365
   